# CIC-IDS2017 Discovery

In [12]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

raw_dir = Path("data/CIC_IDS_2017")
assert raw_dir.exists(), f"Missing CIC-IDS2017 raw directory: {raw_dir}"

csv_files = sorted(raw_dir.glob("*.csv"))
print("Raw directory:", raw_dir)
print("CSV files:", [path.name for path in csv_files])
assert csv_files, f"No CSV files found in {raw_dir}. Place the CIC-IDS2017 daily CSV exports here first."


def read_csv(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, low_memory=False)
    df.columns = [col.strip() for col in df.columns]
    return df


def get_label_col(df: pd.DataFrame):
    for col in ["Label", "label"]:
        if col in df.columns:
            return col
    return None


dfs = {path.name: read_csv(path) for path in csv_files}
print("Loaded files:", len(dfs))

Raw directory: data/CIC_IDS_2017
CSV files: ['Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv', 'Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv', 'Friday-WorkingHours-Morning.pcap_ISCX.csv', 'Monday-WorkingHours.pcap_ISCX.csv', 'Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv', 'Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv', 'Tuesday-WorkingHours.pcap_ISCX.csv', 'Wednesday-workingHours.pcap_ISCX.csv']
Loaded files: 8


In [13]:
file_rows = []
for name, df in dfs.items():
    label_col = get_label_col(df)
    numeric_df = df.select_dtypes(include=[np.number])
    inf_count = int(np.isinf(numeric_df.to_numpy()).sum()) if not numeric_df.empty else 0
    file_rows.append(
        {
            "file": name,
            "rows": len(df),
            "cols": len(df.columns),
            "duplicate_rows": int(df.duplicated().sum()),
            "missing_cells": int(df.isna().sum().sum()),
            "inf_cells": inf_count,
            "label_col": label_col,
            "n_labels": int(df[label_col].nunique(dropna=False)) if label_col else 0,
        }
    )

display(pd.DataFrame(file_rows).sort_values("file"))

for name, df in dfs.items():
    label_col = get_label_col(df)
    if not label_col:
        continue
    print(f"\nTop labels for {name} ({label_col}):")
    display(df[label_col].astype(str).value_counts(dropna=False).head(20).to_frame("count"))


,file,rows,cols,duplicate_rows,missing_cells,inf_cells,label_col,n_labels
0,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,225745,79,2633,4,64,Label,2
1,Friday-WorkingHours-Afternoon-PortScan.pcap_IS...,286467,79,72353,15,727,Label,2
2,Friday-WorkingHours-Morning.pcap_ISCX.csv,191033,79,6888,28,216,Label,2
3,Monday-WorkingHours.pcap_ISCX.csv,529918,79,26935,64,810,Label,1
4,Thursday-WorkingHours-Afternoon-Infilteration....,288602,79,35630,18,396,Label,2
5,Thursday-WorkingHours-Morning-WebAttacks.pcap_...,170366,79,6066,20,250,Label,4
6,Tuesday-WorkingHours.pcap_ISCX.csv,445909,79,24065,201,327,Label,3
7,Wednesday-workingHours.pcap_ISCX.csv,692703,79,81909,1008,1586,Label,6



Top labels for Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv (Label):


,count
Label,
DDoS,128027
BENIGN,97718



Top labels for Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv (Label):


,count
Label,
PortScan,158930
BENIGN,127537


...

,count
Label,
BENIGN,168186
Web Attack � Brute Force,1507
Web Attack � XSS,652
Web Attack � Sql Injection,21



Top labels for Tuesday-WorkingHours.pcap_ISCX.csv (Label):


,count
Label,
BENIGN,432074
FTP-Patator,7938
SSH-Patator,5897



Top labels for Wednesday-workingHours.pcap_ISCX.csv (Label):


,count
Label,
BENIGN,440031
DoS Hulk,231073
DoS GoldenEye,10293
DoS slowloris,5796
DoS Slowhttptest,5499
Heartbleed,11


## Discovery Findings

Từ kết quả discovery chạy được trong notebook, CIC-IDS2017 có các đặc điểm chính sau:

- Dataset được tách thành **8 CSV theo ngày/ca**:
  - `Monday-WorkingHours.pcap_ISCX.csv`
  - `Tuesday-WorkingHours.pcap_ISCX.csv`
  - `Wednesday-workingHours.pcap_ISCX.csv`
  - `Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv`
  - `Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv`
  - `Friday-WorkingHours-Morning.pcap_ISCX.csv`
  - `Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv`
  - `Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv`
- Tất cả 8 file có **cùng schema 79 cột**, không có lệch cột giữa các file.
- Mỗi file đều có nhãn `Label` và không file nào thiếu cột nhãn.

Các vấn đề dữ liệu cần ưu tiên xử lý trước cleaning:

- **Duplicate rows** xuất hiện ở mọi file, mức cao nhất ở `Wednesday-workingHours.pcap_ISCX.csv` với **81,909** dòng trùng, tiếp theo là `Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv` với **72,353** và `Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv` với **35,630**.
- **Missing values** có mặt ở tất cả file, dù số lượng không lớn so với tổng kích thước dataset.
- **Infinite values** cũng xuất hiện ở tất cả file; cao nhất vẫn là `Wednesday-workingHours.pcap_ISCX.csv` với **1,586** giá trị `inf`.
- **Class imbalance rất mạnh** theo ngày và theo loại tấn công:
  - `Monday-WorkingHours.pcap_ISCX.csv` chỉ có `BENIGN`.
  - `Wednesday-workingHours.pcap_ISCX.csv` bị chi phối bởi `DoS Hulk` (**231,073** mẫu) so với `BENIGN` (**440,031** mẫu).
  - `Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv` và `Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv` đều là nhị phân BENIGN vs một attack chính.
  - `Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv` có nhiều subtype attack nhưng số lượng attack rất nhỏ so với BENIGN.
- Có dấu hiệu **label normalization / encoding issue** ở nhãn `Web Attack � Brute Force`, cho thấy cần chuẩn hoá tên nhãn trước khi train.
- Dữ liệu được chia theo ngày, nên split train/val/test phải tránh leakage theo file hoặc theo timestamp.
- Nhiều feature của CICFlowMeter có khả năng chứa cột định danh/leakage như Flow ID, IP, port và timestamp; các cột này không nên đưa trực tiếp vào mô hình.
- Các feature dạng rate vẫn cần kiểm tra riêng khi cleaning vì có thể phát sinh `inf`, `-inf` hoặc `NaN` trong một số dòng.

Tóm lại, discovery cho thấy vấn đề lớn nhất của CIC-IDS2017 trong repo này không phải schema drift, mà là duplicate rows, class imbalance, giá trị vô hạn, và label normalization.

# Initial Cleaning

Clean từng file riêng, chưa gộp dữ liệu ở bước này:

- drop duplicate rows
- loại bỏ leakage columns
- chuẩn hoá label strings
- replace `inf/-inf` bằng `NaN` rồi drop rows có missing
- giữ `constant_cols` để xử lý sau khi gộp file
- giữ output riêng cho từng file để bước split làm sau

In [14]:
drop_cols = [
    "Flow ID",
    "Flow_ID",
    "Source IP",
    "Src_IP",
    "Source Port",
    "Src_Port",
    "Destination IP",
    "Dst_IP",
    "Destination Port",
    "Dst_Port",
    "Timestamp",
]


def normalize_label_series(series: pd.Series) -> pd.Series:
    return (
        series.astype("string")
        .str.strip()
        .str.replace("�", "-", regex=False)
        .str.replace(r"\s+", " ", regex=True)
    )


def clean_single_frame(df: pd.DataFrame):
    report = {}
    df = df.copy()
    df.columns = [col.strip() for col in df.columns]

    report["raw_rows"] = len(df)
    report["raw_cols"] = len(df.columns)
    report["raw_duplicates"] = int(df.duplicated().sum())
    df = df.drop_duplicates().reset_index(drop=True)

    report["dropped_leakage_cols"] = ", ".join(drop_cols)
    df = df.drop(columns=drop_cols, errors="ignore")

    if "Label" in df.columns:
        df["Label"] = normalize_label_series(df["Label"])

    df = df.replace([np.inf, -np.inf], np.nan)
    report["rows_with_missing_before_dropna"] = int(df.isna().any(axis=1).sum())
    report["missing_cells_before_dropna"] = int(df.isna().sum().sum())

    df = df.dropna().reset_index(drop=True)

    constant_cols = [
        col for col in df.columns
        if col != "Label" and df[col].nunique(dropna=False) <= 1
    ]
    report["constant_cols_count"] = len(constant_cols)
    report["constant_cols_preview"] = ", ".join(constant_cols[:8])

    report["final_rows"] = len(df)
    report["final_cols"] = len(df.columns)
    report["final_duplicates"] = int(df.duplicated().sum())

    return df, report


cleaned_frames = {}
cleaning_reports = []

for file_name, df in dfs.items():
    df_clean_i, report = clean_single_frame(df)
    report["file"] = file_name
    cleaned_frames[file_name] = df_clean_i
    cleaning_reports.append(report)

cleaning_report_df = pd.DataFrame(cleaning_reports).sort_values("file")
display(cleaning_report_df)


,raw_rows,raw_cols,raw_duplicates,dropped_leakage_cols,rows_with_missing_before_dropna,missing_cells_before_dropna,constant_cols_count,constant_cols_preview,final_rows,final_cols,final_duplicates,file
0,225745,79,2633,"Flow ID, Flow_ID, Source IP, Src_IP, Source Po...",30,60,10,"Bwd PSH Flags, Fwd URG Flags, Bwd URG Flags, C...",223082,78,1818,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
1,286467,79,72353,"Flow ID, Flow_ID, Source IP, Src_IP, Source Po...",337,674,10,"Bwd PSH Flags, Fwd URG Flags, Bwd URG Flags, C...",213777,78,94255,Friday-WorkingHours-Afternoon-PortScan.pcap_IS...
2,191033,79,6888,"Flow ID, Flow_ID, Source IP, Src_IP, Source Po...",101,202,10,"Bwd PSH Flags, Fwd URG Flags, Bwd URG Flags, C...",184044,78,8006,Friday-WorkingHours-Morning.pcap_ISCX.csv
3,529918,79,26935,"Flow ID, Flow_ID, Source IP, Src_IP, Source Po...",333,666,10,"Bwd PSH Flags, Fwd URG Flags, Bwd URG Flags, C...",502650,78,43819,Monday-WorkingHours.pcap_ISCX.csv
4,288602,79,35630,"Flow ID, Flow_ID, Source IP, Src_IP, Source Po...",182,364,8,"Bwd PSH Flags, Bwd URG Flags, Fwd Avg Bytes/Bu...",252790,78,45160,Thursday-WorkingHours-Afternoon-Infilteration....
5,170366,79,6066,"Flow ID, Flow_ID, Source IP, Src_IP, Source Po...",121,242,10,"Bwd PSH Flags, Fwd URG Flags, Bwd URG Flags, C...",164179,78,8359,Thursday-WorkingHours-Morning-WebAttacks.pcap_...
6,445909,79,24065,"Flow ID, Flow_ID, Source IP, Src_IP, Source Po...",218,436,10,"Bwd PSH Flags, Fwd URG Flags, Bwd URG Flags, C...",421626,78,31912,Tuesday-WorkingHours.pcap_ISCX.csv
7,692703,79,81909,"Flow ID, Flow_ID, Source IP, Src_IP, Source Po...",302,604,10,"Bwd PSH Flags, Fwd URG Flags, Bwd URG Flags, C...",610492,78,25501,Wednesday-workingHours.pcap_ISCX.csv


# Data Split

In [15]:
# Data split for CIC-IDS2017 (file-wise split to reduce leakage)
# Requires: cleaned_frames from Initial Cleaning
# Output: train_df, val_df, test_df

from sklearn.model_selection import GroupShuffleSplit
import pandas as pd

assert "cleaned_frames" in globals(), "cleaned_frames is missing. Run Initial Cleaning first."

# Rebuild one dataframe only for splitting
split_parts = []
for file_name, df_part in cleaned_frames.items():
    tmp = df_part.copy()
    tmp["Source_File"] = file_name
    split_parts.append(tmp)

df_all = pd.concat(split_parts, ignore_index=True)

print("Full cleaned shape before split:", df_all.shape)
print("Files:", df_all["Source_File"].nunique())
print("Labels:", df_all["Label"].nunique())

# 1) Split off test by file
gss1 = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
trainval_idx, test_idx = next(gss1.split(df_all, y=df_all["Label"], groups=df_all["Source_File"]))

trainval_df = df_all.iloc[trainval_idx].reset_index(drop=True)
test_df = df_all.iloc[test_idx].reset_index(drop=True)

# 2) Split train/val by file
gss2 = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, val_idx = next(gss2.split(trainval_df, y=trainval_df["Label"], groups=trainval_df["Source_File"]))

train_df = trainval_df.iloc[train_idx].reset_index(drop=True)
val_df = trainval_df.iloc[val_idx].reset_index(drop=True)

# Drop split-only metadata column if you don't want it downstream
# Keep it for now so you can inspect leakage-safe grouping
print("\nSplit sizes:")
print("train:", train_df.shape)
print("val  :", val_df.shape)
print("test :", test_df.shape)

def label_report(df, name):
    print(f"\n[{name}] label distribution:")
    display(df["Label"].value_counts(dropna=False).to_frame("count"))
    print(f"[{name}] source files:", sorted(df["Source_File"].unique().tolist()))

label_report(train_df, "train")
label_report(val_df, "val")
label_report(test_df, "test")

# Coverage check: warn if some labels are not seen in train
all_labels = set(df_all["Label"].astype(str).unique())
train_labels = set(train_df["Label"].astype(str).unique())
missing_in_train = sorted(all_labels - train_labels)

print("\nLabels missing from train:", missing_in_train)
if missing_in_train:
    print("Warning: these labels will not be learnable by a standard multiclass classifier trained only on train.")


Full cleaned shape before split: (2572640, 79)
Files: 8
Labels: 15

Split sizes:
train: (1787558, 79)
val  : (407126, 79)
test : (377956, 79)

[train] label distribution:


,count
Label,
BENIGN,1584616
DoS Hulk,172846
DoS GoldenEye,10286
FTP-Patator,5931
DoS slowloris,5385
DoS Slowhttptest,5228
SSH-Patator,3219
Infiltration,36
Heartbleed,11


[train] source files: ['Monday-WorkingHours.pcap_ISCX.csv', 'Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv', 'Tuesday-WorkingHours.pcap_ISCX.csv', 'Wednesday-workingHours.pcap_ISCX.csv']

[val] label distribution:


,count
Label,
BENIGN,277164
DDoS,128014
Bot,1948


[val] source files: ['Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv', 'Friday-WorkingHours-Morning.pcap_ISCX.csv']

[test] label distribution:


,count
Label,
BENIGN,285119
PortScan,90694
Web Attack - Brute Force,1470
Web Attack - XSS,652
Web Attack - Sql Injection,21


[test] source files: ['Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv', 'Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv']

Labels missing from train: ['Bot', 'DDoS', 'PortScan', 'Web Attack - Brute Force', 'Web Attack - Sql Injection', 'Web Attack - XSS']
